# 10 · Gradient-Based Geometry Inversion

*Use automatic differentiation for efficient geometry refinement and local error bars.*

**Learning objectives**

- explain automatic differentiation at a practical level
- run differentiable variable projection with JAX
- interpret parameter scaling and compilation cost
- judge when Gauss–Newton covariance is credible

**Prerequisites:** Chapter 09; `geodef[jax]`  
**Estimated time:** about 60 minutes

> **Optional dependency:** install `geodef[jax]`.

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

Finite-difference geometry searches repeatedly evaluate nearby models merely
to estimate a slope. Automatic differentiation propagates derivatives through
the analytic forward model directly. It can accelerate multi-parameter search,
but fast local curvature is still only a local uncertainty approximation.


## Theory deepening: derivatives and local curvature

Automatic differentiation applies the chain rule to the executed forward
calculation. Forward mode is efficient for few inputs; reverse mode is
efficient for scalar objectives with many inputs. JAX composes both with the
differentiable elastic kernels and variable-projection solve.

L-BFGS-B uses gradients plus a compact inverse-Hessian approximation. Parameter
scaling matters because meters and degrees otherwise produce very different
gradient magnitudes. At the optimum, a Gauss–Newton covariance uses local
Jacobian curvature. It is credible only when the selected basin is unique,
residuals are locally linear, and bounds are inactive. Chapter 14 replaces this
local Gaussian picture with posterior sampling.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

geodef.backend.set_backend("jax")   # float64 by default

rng = np.random.default_rng(0)

# --- Recurring teaching scenario (identical to notebook 10) ----------------
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.data.gnss(lon=glon, lat=glat, east=_zero, north=_zero, up=_zero, sigma_east=_one, sigma_north=_one, sigma_up=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.data.gnss(
    lon=glon, lat=glat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_sta, sigma), sigma_north=np.full(n_sta, sigma), sigma_up=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, backend = {geodef.backend.get_backend()}")

## Variable projection: linear inside, nonlinear outside

The trick (same as notebook 10, now differentiable end to end): for any trial
geometry $\boldsymbol\theta$, the *slip* is a **linear** problem, solved
exactly by regularized least squares,

$$\mathbf m^*(\boldsymbol\theta) = \left(\mathbf G(\boldsymbol\theta)^T
\mathbf W \mathbf G(\boldsymbol\theta) + \lambda\,\mathbf L^T\mathbf L
\right)^{-1}\mathbf G(\boldsymbol\theta)^T \mathbf W \mathbf d ,$$

so the objective reduces to a function of geometry alone:
$\Phi(\boldsymbol\theta) = \mathbf r^T \mathbf W\, \mathbf r$ with
$\mathbf r = \mathbf d - \mathbf G(\boldsymbol\theta)\,
\mathbf m^*(\boldsymbol\theta)$. JAX differentiates through the whole
expression — Okada kernel, patch grid, and the linear solve included — so
`geodef.invert.geometry_search` hands L-BFGS-B the exact
$\nabla_{\!\boldsymbol\theta}\Phi$.

The geometry vector is
$\boldsymbol\theta = [e_0, n_0, \text{depth}, \text{strike}, \text{dip},
\text{length}, \text{width}]$ in a local Cartesian frame anchored at
`(ref_lat, ref_lon)`; the `free` argument picks which entries to optimize.

The first call JIT-compiles the kernel (tens of seconds); repeated calls with
the same problem shapes — multi-start, exercises below — reuse the
compilation and run in milliseconds per iteration.

In [ ]:
# Single unknown, exactly notebook 10's problem: recover dip from a bad start
theta0 = np.array([0.0, 0.0, 25e3, 315.0, 30.0, 180e3, 90e3])  # start dip 30

res_dip = geodef.invert.geometry_search(
    theta0, gnss,
    ref_lat=-2.0, ref_lon=100.0,
    free=["dip"],
    bounds={"dip": (5.0, 45.0)},
    n_length=nL, n_width=nW,
    components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
sig_dip = np.sqrt(res_dip.theta_cov[0, 0])
print(f"recovered dip: {res_dip.theta[4]:.2f} ± {sig_dip:.2f} deg   (true 15.0)")
print(f"reduced chi^2: {res_dip.reduced_chi2:.3f},  iterations: {res_dip.n_iterations}")

## Several parameters at once

This is where gradients earn their keep. A grid over dip *and* depth at
notebook 10's resolution would need hundreds of full inversions; the
gradient-based search converges in a few dozen iterations, and the
Gauss-Newton covariance gives first-order error bars on both parameters.

In [ ]:
theta0 = np.array([0.0, 0.0, 35e3, 315.0, 30.0, 180e3, 90e3])  # both wrong

res = geodef.invert.geometry_search(
    theta0, gnss,
    ref_lat=-2.0, ref_lon=100.0,
    free=["dip", "depth"],
    bounds={"dip": (5.0, 45.0), "depth": (10e3, 60e3)},
    n_length=nL, n_width=nW,
    components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
sig = np.sqrt(np.diag(res.theta_cov))
print(f"recovered dip:   {res.theta[4]:8.2f} ± {sig[0]:.2f} deg   (true 15.0)")
print(f"recovered depth: {res.theta[2]/1e3:8.2f} ± {sig[1]/1e3:.2f} km    (true 25.0)")
print(f"reduced chi^2:   {res.reduced_chi2:8.3f}")

In [ ]:
# Recovered slip at the optimal geometry, next to the truth
best_fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=res.theta[2], strike=315.0, dip=res.theta[4],
    length=180e3, width=90e3, n_length=nL, n_width=nW,
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True (dip 15, depth 25 km)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(best_fault, res.slip, ax=ax2, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax,
                 title=f"Recovered (dip {res.theta[4]:.1f}, "
                       f"depth {res.theta[2]/1e3:.1f} km)",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Checkpoint questions

**Why can the first JAX call be slower?**

<details><summary>Answer</summary>



It includes tracing and compilation; later calls reuse compiled code.



</details>

**What does automatic differentiation remove?**

<details><summary>Answer</summary>



Finite-difference truncation and repeated perturbed evaluations, not objective nonconvexity.



</details>

**When is Gauss–Newton covariance misleading?**

<details><summary>Answer</summary>



With nonlinearity, multiple modes, active bounds, or misspecified noise.



</details>


## Common mistakes

- Benchmarking compile and execution time as one number hides reuse behavior.
- Leaving float64 disabled can change geodetic cancellation accuracy.
- Trusting a fast local optimizer without a basin scan repeats Chapter 09's mistake faster.


## Recap

- Autodiff provides exact derivatives of the implemented computation.
- Gradient search improves efficiency but does not create identifiability.
- Curvature-based geometry errors are local and conditional.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/10_gradient_geometry_solutions.ipynb`.

1. Recover three, then five free parameters and compare iterations.
2. Compare compile time with repeated execution time.
3. Change parameter scales and observe optimizer behavior.
4. Challenge: validate one gradient component with centered finite differences.


## Further reading

- Griewank & Walther (2008), automatic differentiation.
- Nocedal & Wright (2006), L-BFGS-B and optimization.
- Golub & Pereyra (1973), differentiable variable projection.
